# **Import**


In [8]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from data_loader import Dataset
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

# **Build From Scratch**


In [9]:
class Node():

    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, benefit=None, value=None):

        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.benefit = benefit

        self.value = value

In [10]:
def _entropy(y):

    class_labels = np.unique(y)
    entropy = 0

    for k in class_labels:

        p_k = len(y[y == k]) / len(y)
        entropy += p_k * np.log2(p_k)

    return -entropy

In [11]:
class CustomDecisionTree():

    def __init__(self, min_samples_split=5, max_depth=5):

        self.root = None

        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def _build_tree(self, dataset, curr_depth=0):

        X, y = dataset[:, :-1], dataset[:, -1]

        num_samples, num_features = np.shape(X)

        if num_samples >= self.min_samples_split and curr_depth <= self.max_depth:

            best_decision = self._get_best_decision(dataset, num_features)

            if best_decision["benefit"] > 0:

                child_l = self._build_tree(best_decision["dataset_left"], curr_depth + 1)
                child_r = self._build_tree(best_decision["dataset_right"], curr_depth + 1)

                return Node(best_decision["feature_idx"], best_decision["threshold"], child_l, child_r, best_decision["benefit"])

        leaf_value = self._calculate_leaf_value(y)

        return Node(value = leaf_value)

    def _get_best_decision(self, dataset, num_features, k=10):

        best_decision = {}

        max_benefit = -float("inf")

        for feature_idx in range(num_features):

            feature_values = dataset[:, feature_idx]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds[::k]:

                dataset_left, dataset_right = self._split(dataset, feature_idx, threshold)

                if len(dataset_left) > 0 and len(dataset_right) > 0:

                    y, y_left, y_right = dataset[:, -1], dataset_left[:, -1], dataset_right[:, -1]

                    curr_benefit = self._compute_benefit(y, y_left, y_right)

                    if curr_benefit > max_benefit:
                        best_decision["feature_idx"] = feature_idx
                        best_decision["threshold"] = threshold
                        best_decision["dataset_left"] = dataset_left
                        best_decision["dataset_right"] = dataset_right
                        best_decision["benefit"] = curr_benefit
                        max_benefit = curr_benefit

        return best_decision

    def _split(self, dataset, feature_idx, threshold):

        dataset_left = np.array([row for row in dataset if row[feature_idx] <= threshold])
        dataset_right = np.array([row for row in dataset if row[feature_idx] > threshold])

        return dataset_left, dataset_right

    def _compute_benefit(self, parent, child_l, child_r):

        p_l = len(child_l) / len(parent)
        p_r = len(child_r) / len(parent)

        if self.problem_type == "classification":
            return _entropy(parent) - (p_l * _entropy(child_l) + p_r * _entropy(child_r))
        else:
            return np.var(parent) - (p_l * np.var(child_l) + p_r * np.var(child_r))

    def _calculate_leaf_value(self, y):

        if self.problem_type == "classification":
            y = list(y)
            return max(y, key=y.count)
        else:
            return np.mean(y)

    def fit(self, X, y):

        if len(set(y)) <= 20:
            self.problem_type = "classification"
        else:
            self.problem_type = "regression"

        dataset = np.concatenate((X, y.reshape(-1, 1)), axis=1)

        self.root = self._build_tree(dataset)

    def predict(self, X):

        predictions = [self._predict_for_observ(x, self.root) for x in X]

        return predictions

    def _predict_for_observ(self, x, tree):

        if tree.value is not None:
            return tree.value

        feature_val = x[tree.feature_idx]

        if feature_val <= tree.threshold:
            return self._predict_for_observ(x, tree.left)

        else:
            return self._predict_for_observ(x, tree.right)


# **Load and Split**

In [12]:
dataset_m = Dataset("multiclass classification")
X_train_m, X_test_m, y_train_m, y_test_m = dataset_m.load_split_data()

dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()

dataset_r = Dataset("regression")
X_train_r, X_test_r, y_train_r, y_test_r = dataset_r.load_split_data()

# **Train, Test and Compare**

In [13]:
custom_model_m = CustomDecisionTree()
custom_model_m.fit(X_train_m, y_train_m)
y_pred_custom_m = custom_model_m.predict(X_test_m)
print(f"Custom Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_custom_m):.3f}")

custom_model_b = CustomDecisionTree()
custom_model_b.fit(X_train_b, y_train_b)
y_pred_custom_b = custom_model_b.predict(X_test_b)
print(f"Custom Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_custom_b):.3f}")

custom_model_r = CustomDecisionTree()
custom_model_r.fit(X_train_r, y_train_r)
y_pred_custom_r = custom_model_r.predict(X_test_r)
print(f"Custom Regression MSE: {mean_squared_error(y_test_r, y_pred_custom_r):.3f}")

Custom Multiclass Classification Accuracy: 0.933
Custom Binary Classification Accuracy: 0.971
Custom Regression MSE: 0.494


In [14]:
sklearn_model_m = DecisionTreeClassifier()
sklearn_model_m.fit(X_train_m, y_train_m)
y_pred_sklearn_m = sklearn_model_m.predict(X_test_m)
print(f"Scikit-learn Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_sklearn_m):.3f}")

sklearn_model_b = DecisionTreeClassifier()
sklearn_model_b.fit(X_train_b, y_train_b)
y_pred_sklearn_b = sklearn_model_b.predict(X_test_b)
print(f"Scikit-learn Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_sklearn_b):.3f}")

sklearn_model_r = DecisionTreeRegressor()
sklearn_model_r.fit(X_train_r, y_train_r)
y_pred_sklearn_r = sklearn_model_r.predict(X_test_r)
print(f"Scikit-learn Regression MSE: {mean_squared_error(y_test_r, y_pred_sklearn_r):.3f}")

Scikit-learn Multiclass Classification Accuracy: 0.967
Scikit-learn Binary Classification Accuracy: 0.949
Scikit-learn Regression MSE: 0.477
